# Phase 9: Phân Tích Lỗi Chi Tiết (Error Analysis)

Notebook này thực hiện phân loại lỗi của các mô hình trích xuất thông tin biên lai tiếng Việt theo bộ taxonomy tiêu chuẩn từ E1 đến E11:

- **E1 (OCR Miss)**: OCR bỏ sót text hoàn toàn.
- **E2 (OCR Wrong)**: Nhận dạng sai dấu/ký tự tiếng Việt (ví dụ: `Thành phố` -> `Thanh pho`).
- **E3 (Wrong Total)**: Nhầm lẫn giữa subtotal (cộng tiền hàng) và total (tổng thanh toán).
- **E4 (Wrong Date)**: Lấy nhầm ngày giờ in hóa đơn/ngày thu ngân thay vì ngày giao dịch chính thức.
- **E5 (Multi-line Address)**: Trích xuất thiếu dòng địa chỉ (chỉ lấy 1 dòng trong khi địa chỉ gồm nhiều dòng).
- **E6 (Format Error)**: Donut sinh chuỗi lỗi thẻ tag hoặc lỗi cú pháp JSON.
- **E7 (Hallucination)**: Donut tự sinh thông tin không có trong ảnh.
- **E8 (Layout BIO broken)**: LayoutXLM gán BIO rời rạc không ghép được thực thể hoàn chỉnh.
- **E9 (Wrong Reading Order)**: Thuật toán sắp xếp dòng text bị sai khiến LayoutXLM phân loại nhầm context.
- **E10 (Normalization Error)**: Trích xuất text đúng nhưng chuẩn hóa sai.
- **E11 (Missing Ground Truth)**: Ground truth bị thiếu/rỗng trường thông tin nhưng mô hình thực tế trích xuất đúng (False Positive nhưng thực chất là GT thiếu nhãn).

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from rapidfuzz.distance import Levenshtein

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'

## 1. Định nghĩa Hàm Phân Loại Lỗi Tự Động

In [ ]:
def classify_error(pred_val, gt_val, field_name, model_type="ocr_based", raw_pred_val=None):
    """
    Phân loại lỗi dựa trên so sánh giữa dự đoán đã chuẩn hóa (pred_val) và nhãn chuẩn hóa (gt_val).
    """
    pred_val = str(pred_val).strip() if pred_val is not None else ""
    gt_val = str(gt_val).strip() if gt_val is not None else ""
    raw_pred_val = str(raw_pred_val).strip() if raw_pred_val is not None else pred_val
    
    if pred_val == gt_val:
        return "No Error"
        
    # E11: Ground truth bị thiếu nhưng mô hình tìm ra được
    if gt_val == "" and pred_val != "":
        return "E11 (Missing GT / False Positive)"
        
    # Mô hình bỏ sót thông tin (Dự đoán rỗng trong khi Ground Truth có giá trị)
    if pred_val == "" and gt_val != "":
        if model_type == "donut":
            return "E7 (Donut Miss/Hallucination)"
        else:
            # OCR-based: có thể do OCR hụt chữ hoặc Layout phân loại sai
            return "E1 (OCR Miss) / E8 (BIO Broken)"

    # Đo độ tương đồng để phân biệt lỗi chính tả (OCR Wrong/Normalization) và lỗi sai thông tin hoàn toàn
    lev_dist = Levenshtein.distance(pred_val, gt_val)
    max_len = max(len(pred_val), len(gt_val))
    similarity = 1 - (lev_dist / max_len) if max_len > 0 else 1.0

    # Lỗi sai chính tả nhẹ (ví dụ thiếu dấu, sai 1-2 ký tự do OCR hoặc Normalize)
    if similarity >= 0.75:
        if similarity < 1.0:
            # So sánh raw prediction và gt_val chưa normalize nếu có
            return "E2 (OCR Wrong) / E10 (Normalization Error)"

    # Lỗi sai thông tin hoàn toàn (similarity thấp)
    if field_name == "total":
        return "E3 (Wrong Total / Subtotal Confusion)"
    elif field_name == "date":
        return "E4 (Wrong Date / Timestamp Confusion)"
    elif field_name == "address":
        # Nếu trích xuất được địa chỉ nhưng quá ngắn so với GT
        if len(pred_val) < 0.5 * len(gt_val):
            return "E5 (Multi-line Address / Incomplete)"
        return "E8 (BIO Broken) / E9 (Wrong Reading Order)"
    elif field_name == "store_name":
        return "E8 (BIO Broken) / E2 (OCR Wrong)"
        
    return "General Classification Error"

## 2. Tạo Dữ Liệu Giả Lập (Mock Predictions) Để Minh Họa
*(Nếu chưa có kết quả chạy thực tế từ pipeline)*

In [ ]:
def generate_mock_predictions():
    # Tạo danh sách các test samples
    mock_ground_truth = [
        {
            "id": "rec_01",
            "target": {"store_name": "CÔNG TY CỔ PHẦN CON CƯNG", "date": "2023-11-15", "total": "450000", "address": "101 Nguyễn Thị Minh Khai, Q3, TP.HCM"}
        },
        {
            "id": "rec_02",
            "target": {"store_name": "Circle K Việt Nam", "date": "2023-12-01", "total": "75000", "address": "20 Bùi Thị Xuân, Quận 1, TPHCM"}
        },
        {
            "id": "rec_03",
            "target": {"store_name": "NHÀ SÁCH PHƯƠNG NAM", "date": "2023-10-20", "total": "120000", "address": "Đường Lê Lợi, Quận 1, TP Hồ Chí Minh"}
        },
        {
            "id": "rec_04",
            "target": {"store_name": "LOTTERIA VIETNAM", "date": "2023-09-05", "total": "185000", "address": "Lotte Mart Quận 7, TP.HCM"}
        },
        {
            "id": "rec_05",
            "target": {"store_name": "", "date": "2023-08-12", "total": "320000", "address": "12 Hà Huy Tập, Đà Nẵng"} # Thiếu store_name ở GT
        }
    ]
    
    # Giả lập dự đoán từ Baseline
    baseline_preds = [
        # Đúng hết
        {"id": "rec_01", "normalized_prediction": {"store_name": "CON CƯNG", "date": "2023-11-15", "total": "450000", "address": "101 Nguyễn Thị Minh Khai, Q3, TP.HCM"}},
        # Circle K: sai ngày (lấy ngày in hóa đơn), sai total (nhầm subtotal)
        {"id": "rec_02", "normalized_prediction": {"store_name": "Circle K", "date": "2023-12-02", "total": "70000", "address": "20 Bùi Thị Xuân, Quận 1"}},
        # Phương Nam: OCR sai dấu chữ Phương Nam -> Phuong Nam, thiếu dòng địa chỉ
        {"id": "rec_03", "normalized_prediction": {"store_name": "NHA SACH PHUONG NAM", "date": "2023-10-20", "total": "120000", "address": "Đường Lê Lợi"}},
        # Lotteria: Bỏ sót không nhận diện được ngày
        {"id": "rec_04", "normalized_prediction": {"store_name": "LOTTERIA VIETNAM", "date": "", "total": "185000", "address": "Lotte Mart Quận 7, TP.HCM"}},
        # GT rỗng store_name nhưng baseline đoán ra
        {"id": "rec_05", "normalized_prediction": {"store_name": "CỬA HÀNG TIỆN LỢI", "date": "2023-08-12", "total": "320000", "address": "12 Hà Huy Tập"}}
    ]
    
    # Giả lập dự đoán từ Donut
    donut_preds = [
        {"id": "rec_01", "normalized_prediction": {"store_name": "CÔNG TY CỔ PHẦN CON CƯNG", "date": "2023-11-15", "total": "450000", "address": "101 Nguyễn Thị Minh Khai, Q3, TP.HCM"}},
        # Circle K: Nhầm total, Hallucination tự sinh địa chỉ quận 3
        {"id": "rec_02", "normalized_prediction": {"store_name": "Circle K Việt Nam", "date": "2023-12-01", "total": "70000", "address": "20 Bùi Thị Xuân, Quận 3, TPHCM"}},
        # Phương Nam: Đúng hết
        {"id": "rec_03", "normalized_prediction": {"store_name": "NHÀ SÁCH PHƯƠNG NAM", "date": "2023-10-20", "total": "120000", "address": "Đường Lê Lợi, Quận 1, TP Hồ Chí Minh"}},
        # Lotteria: Lỗi tag format của Donut làm mất trường date
        {"id": "rec_04", "normalized_prediction": {"store_name": "LOTTERIA VIETNAM", "date": "", "total": "185000", "address": "Lotte Mart Quận 7, TP.HCM"}},
        {"id": "rec_05", "normalized_prediction": {"store_name": "TIỆN ÍCH", "date": "2023-08-12", "total": "320000", "address": "12 Hà Huy Tập, Đà Nẵng"}}
    ]
    
    # Giả lập dự đoán từ LayoutXLM
    layoutxlm_preds = [
        {"id": "rec_01", "normalized_prediction": {"store_name": "CÔNG TY CỔ PHẦN CON CƯNG", "date": "2023-11-15", "total": "450000", "address": "101 Nguyễn Thị Minh Khai, Q3, TP.HCM"}},
        # Circle K: Đúng hết
        {"id": "rec_02", "normalized_prediction": {"store_name": "Circle K Việt Nam", "date": "2023-12-01", "total": "75000", "address": "20 Bùi Thị Xuân, Quận 1, TPHCM"}},
        # Phương Nam: Layout BIO broken làm đứt đoạn địa chỉ
        {"id": "rec_03", "normalized_prediction": {"store_name": "NHÀ SÁCH PHƯƠNG NAM", "date": "2023-10-20", "total": "120000", "address": "Đường Lê Lợi"}},
        # Lotteria: Đúng hết
        {"id": "rec_04", "normalized_prediction": {"store_name": "LOTTERIA VIETNAM", "date": "2023-09-05", "total": "185000", "address": "Lotte Mart Quận 7, TP.HCM"}},
        # GT rỗng store_name nhưng đoán đúng thực tế
        {"id": "rec_05", "normalized_prediction": {"store_name": "CỬA HÀNG TIỆN LỢI", "date": "2023-08-12", "total": "320000", "address": "12 Hà Huy Tập, Đà Nẵng"}}
    ]
    
    return mock_ground_truth, {
        "Baseline": baseline_preds,
        "Donut": donut_preds,
        "LayoutXLM": layoutxlm_preds
    }

## 3. Thực Hiện Phân Tích Lỗi Chi Tiết

In [ ]:
# Load predictions và ground truth
# Trong thực tế, bạn sẽ load từ file:
# preds_path = Path("../outputs/predictions/")

gts, all_preds = generate_mock_predictions()

records = []
for model_name, preds in all_preds.items():
    model_type = "donut" if model_name == "Donut" else "ocr_based"
    
    # Map ID -> prediction
    pred_map = {p["id"]: p["normalized_prediction"] for p in preds}
    
    for gt_item in gts:
        sample_id = gt_item["id"]
        gt_target = gt_item["target"]
        pred_target = pred_map.get(sample_id, {"store_name": "", "date": "", "total": "", "address": ""})
        
        for field in ["store_name", "date", "total", "address"]:
            gt_val = gt_target.get(field, "")
            pred_val = pred_target.get(field, "")
            
            err_cat = classify_error(pred_val, gt_val, field, model_type=model_type)
            
            records.append({
                "model": model_name,
                "sample_id": sample_id,
                "field": field,
                "ground_truth": gt_val,
                "prediction": pred_val,
                "error_type": err_cat
            })

df_errors = pd.DataFrame(records)
df_errors.head(10)

## 4. Báo Cáo Thống Kê Tần Suất Lỗi Của Từng Mô Hình

In [ ]:
# Lọc ra các dòng có lỗi thực sự (khác "No Error")
df_only_errors = df_errors[df_errors["error_type"] != "No Error"]

# Tính toán tỷ lệ phần trăm các loại lỗi của mỗi mô hình
summary = df_only_errors.groupby(["model", "error_type"]).size().reset_index(name="count")
total_errors = df_only_errors.groupby("model").size().reset_index(name="total")
summary = summary.merge(total_errors, on="model")
summary["percentage"] = (summary["count"] / summary["total"]) * 100

print("=== BẢNG THỐNG KÊ CHI TIẾT TẦN SUẤT LỖI CỦA CÁC MÔ HÌNH ===")
pivot_summary = summary.pivot(index="error_type", columns="model", values="count").fillna(0).astype(int)
pivot_summary

## 5. Trực Quan Hóa Lỗi (Visualizing Error Taxonomy)

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(
    data=summary,
    x="error_type",
    y="percentage",
    hue="model",
    palette="muted"
)
plt.title("So Sánh Tỷ Lệ Các Loại Lỗi Giữa Các Mô Hình", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Loại lỗi (Taxonomy)", fontsize=12, labelpad=10)
plt.ylabel("Tỷ lệ phần trăm trên tổng số lỗi (%)", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Mô hình")
plt.tight_layout()
plt.show()

## 6. Trực Quan Hóa Chi Tiết Lỗi Theo Từng Trường Thông Tin (Field-wise Error)

In [ ]:
plt.figure(figsize=(14, 7))
g = sns.catplot(
    data=df_only_errors,
    x="field",
    hue="error_type",
    col="model",
    kind="count",
    height=5,
    aspect=0.8,
    palette="Set2"
)
g.set_axis_labels("Trường thông tin", "Số lượng lỗi")
g.set_titles("Mô hình: {col_name}")
plt.tight_layout()
plt.show()

## 7. Đề Xuất Hướng Khắc Phục Lỗi

Dựa trên phân phối lỗi ở trên, ta có thể đưa ra các giải pháp cải tiến:

1. **Nếu lỗi E1 (OCR Miss) / E2 (OCR Wrong) chiếm ưu thế ở LayoutXLM:**
   - Cần cải thiện PaddleOCR (nâng ngưỡng phát hiện, tuning parameters) hoặc finetune VietOCR với font chữ hóa đơn.
2. **Nếu lỗi E3 (Wrong Total) / E4 (Wrong Date) chiếm ưu thế:**
   - Cần bổ sung các đặc trưng cấu trúc (spatial layout) tốt hơn cho LayoutXLM, hoặc tinh chỉnh regex heurictis của Baseline.
3. **Nếu lỗi E7 (Donut Hallucination) chiếm ưu thế:**
   - Tăng cường dữ liệu (data augmentation) bằng các kỹ thuật tạo ảnh giả lập hoặc huấn luyện Donut thêm epoch với learning rate nhỏ hơn.